# EDA

### Goal of this notebook:
1. Audit the merged NHANES adults dataset
2. Identify candidate target variables for fatigue / low energy / sleep / depression proxies
3. Build a clinically plausible core feature set
4. Assess missingness, subgroup coverage, and feature usefulness
5. Produce a recommendation for next modeling / MVP steps

## Import & Settings

In [4]:
import sys
print(sys.executable)

c:\Users\Inbal\Desktop\PM Inbal\aipm_course_exercises\halfFull\.venv\Scripts\python.exe


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [6]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults.csv")

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
display(df.head())

Shape: (7437, 371)


C:\Users\Inbal\AppData\Local\Temp\ipykernel_19184\357236.py:3: DtypeWarning: Columns (0: medication_18, 1: medication_19, 2: medication_20, 3: medication_21, 4: medication_22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,SEQN,RIAGENDR,RIDAGEYR,RIDRETH3,RIDRETH1,RIDEXPRG,DMDEDUC2,DMDMARTZ,DMDBORN4,INDFMPIR,WTMECPRP,WTINTPRP,SDMVPSU,SDMVSTRA,gender,ethnicity,education,marital_status,pregnancy_status,country_of_birth,calories,protein,carbs,fat,iron,vitamin_b12,vitamin_d,folate,magnesium,zinc,height_cm,weight_kg,bmi,waist_cm,hip_cm,sbp_1,sbp_2,sbp_3,dbp_1,dbp_2,dbp_3,pulse_1,pulse_2,pulse_3,liver_cap_dbm,liver_stiffness_kpa,liver_exam_status,liver_valid_measures,liver_stiffness_iqr_ratio,oral_gum_problem_yesno,oral_decayed_teeth_yesno,oral_hygiene_issue_yesno,any_implant,any_root_caries,any_root_restoration_caries,fasting_hours_part,fasting_minutes_part,fasting_glucose_mg_dl,insulin_uU_ml,total_cholesterol_mg_dl,hdl_cholesterol_mg_dl,triglycerides_mg_dl,ferritin_ng_ml,serum_iron_ug_dl,tibc_ug_dl,transferrin_saturation_pct,transferrin_receptor_mg_l,uacr_mg_g,serum_creatinine_mg_dl,bun_mg_dl,alt_u_l,ast_u_l,ggt_u_l,alp_u_l,total_bilirubin_mg_dl,serum_albumin_g_dl,alq111___ever_had_a_drink_of_any_kind_of_alcohol,alq121___past_12_mo_how_often_drink_alcoholic_bev,alq130___avg_#_alcoholic_drinks/day___past_12_mos,alq142___#_days_have_4_or_5_drinks/past_12_mos,alq270___#_times_4_5_drinks_in_2hrs/past_12_mos,alq280___#_times_8+_drinks_in_1_day/past_12_mos,alq290___#_times_12+_drinks_in_1_day/past_12_mos,alq151___ever_have_4/5_or_more_drinks_every_day?,alq170___past_30_days_#_times_4_5_drinks_on_an_oc,bpq020___ever_told_you_had_high_blood_pressure,bpq030___told_had_high_blood_pressure___2+_times,bpd035___age_told_had_hypertension,bpq040a___taking_prescription_for_hypertension,bpq050a___now_taking_prescribed_medicine_for_hbp,bpq080___doctor_told_you___high_cholesterol_level,bpq060___ever_had_blood_cholesterol_checked,bpq070___when_blood_cholesterol_last_checked,bpq090d___told_to_take_prescriptn_for_cholesterol,bpq100d___now_taking_prescribed_medicine,cdq001___sp_ever_had_pain_or_discomfort_in_chest,cdq002___sp_get_it_walking_uphill_or_in_a_hurry,cdq003___during_an_ordinary_pace_on_level_ground,cdq004___if_so_does_sp_continue_or_slow_down,cdq005___does_standing_relieve_pain/discomfort,...,rxqseen___medicine_container_seen_by_interviewer,rxddays___number_of_days_taken_medicine,rxdrsc1___icd_10_cm_code_1,rxdrsc2___icd_10_cm_code_2,rxdrsc3___icd_10_cm_code_3,rxdrsd1___icd_10_cm_code_1_description,rxdrsd2___icd_10_cm_code_2_description,rxdrsd3___icd_10_cm_code_3_description,rxdcount___number_of_prescription_medicines_taken,slq300___usual_sleep_time_on_weekdays_or_workdays,slq310___usual_wake_time_on_weekdays_or_workdays,sld012___sleep_hours___weekdays_or_workdays,slq320___usual_sleep_time_on_weekends,slq330___usual_wake_time_on_weekends,sld013___sleep_hours___weekends,slq030___how_often_do_you_snore?,slq040___how_often_do_you_snort_or_stop_breathing,slq050___ever_told_doctor_had_trouble_sleeping?,smq020___smoked_at_least_100_cigarettes_in_life,smd030___age_started_smoking_cigarettes_regularly,smq040___do_you_now_smoke_cigarettes?,smq050q___how_long_since_quit_smoking_cigarettes,smq050u___unit_of_measure_(day/week/month/year),smd057___#_cigarettes_smoked_per_day_when_quit,smq078___how_soon_after_waking_do_you_smoke,smd641___#_days_smoked_cigs_during_past_30_days,smd650___avg_#_cigarettes/day_during_past_30_days,smd100fl___cigarette_filter_type,smd100mn___cigarette_menthol_indicator,smq670___tried_to_quit_smoking,smq621___cigarettes_smoked_in_entire_life,smd630___age_first_smoked_whole_cigarette,smaquex2___questionnaire_mode_flag,whd010___current_self_reported_height_(inches),whd020___current_self_reported_weight_(pounds),whq030___how_do_you_consider_your_weight,"whq040___like_to_weigh_more,_less_or_same",whd050___self_reported_weight___1_yr_ago_(pounds),whq060___weight_change_intentional,whq070___tried_to_lose_weight_in_past_year,whd080a___ate_less_to_lose_weight,whd080b___switched_to_foods_with_lower_calories,whd080c___ate_less_fat_to_lose_weight,whd080d___exercised_to_lose_weight,whd080e___skipped_meals,whd080f___ate_diet_foods_or_products,whd080g___used_a_liq

In [7]:
def dataset_overview(dataframe):
    overview = pd.DataFrame({
        "column": dataframe.columns,
        "dtype": dataframe.dtypes.astype(str).values,
        "missing_n": dataframe.isna().sum().values,
        "missing_pct": (dataframe.isna().mean().values * 100).round(2),
        "n_unique": dataframe.nunique(dropna=True).values
    }).sort_values("missing_pct", ascending=False)
    return overview

overview = dataset_overview(df)
display(overview.head(30))

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Numeric columns:", df.select_dtypes(include=np.number).shape[1])
print("Non-numeric columns:", df.select_dtypes(exclude=np.number).shape[1])

,column,dtype,missing_n,missing_pct,n_unique
301,smq621___cigarettes_smoked_in_entire_life,float64,7437,100.00,0
302,smd630___age_first_smoked_whole_cigarette,float64,7437,100.00,0
174,mcq151___age_in_years_at_first_menstrual_period,float64,7437,100.00,0
173,mcq149___menstrual_periods_started_yet?,float64,7437,100.00,0
365,medication_22,str,7435,99.97,2
364,medication_21,str,7435,99.97,2
363,medication_20,str,7434,99.96,3
210,mcq230c___what_kind_of_cancer_third_mention,float64,7432,99.93,4
362,medication_19,str,7432,99.93,5
196,mcq510b___liver_condition_non_alcoholic_fatty_liver,float64,7431,99.92,1


Rows: 7437
Columns: 371
Numeric columns: 330
Non-numeric columns: 41


## Rename column names (domain demographics) and remove doubles

In [8]:
from pathlib import Path
import pandas as pd

# Sicherheitskopie
df_clean = df.copy()

# 1) Kryptische NHANES-Spalten droppen, wenn bereits eine lesbare Version existiert
drop_if_readable_exists = {
    "RIAGENDR": "gender",
    "RIDRETH1": "ethnicity",
    "RIDRETH3": "ethnicity",
    "DMDEDUC2": "education",
    "DMDMARTZ": "marital_status",
    "RIDEXPRG": "pregnancy_status",
    "DMDBORN4": "country_of_birth",
}

cols_to_drop = [
    raw_col
    for raw_col, readable_col in drop_if_readable_exists.items()
    if raw_col in df_clean.columns and readable_col in df_clean.columns
]

# 2) Spalten umbenennen, für die es noch keine lesbare Version gibt
rename_map = {
    "RIDAGEYR": "age_years",
    "INDFMPIR": "income_poverty_ratio",
    "WTMECPRP": "mec_exam_weight",
    "WTINTPRP": "interview_weight",
    "SDMVPSU": "survey_psu",
    "SDMVSTRA": "survey_stratum",
}

# Nur umbenennen, wenn die Originalspalte existiert und der Zielname noch nicht existiert
rename_map_final = {
    old: new
    for old, new in rename_map.items()
    if old in df_clean.columns and new not in df_clean.columns
}

# Anwenden
df_clean = df_clean.drop(columns=cols_to_drop).rename(columns=rename_map_final)

print("Gedroppt:")
print(cols_to_drop)

print("\nUmbenannt:")
print(rename_map_final)

print("\nNeue Shape:", df_clean.shape)
display(df_clean.head())

Gedroppt:
['RIAGENDR', 'RIDRETH1', 'RIDRETH3', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDBORN4']

Umbenannt:
{'RIDAGEYR': 'age_years', 'INDFMPIR': 'income_poverty_ratio', 'WTMECPRP': 'mec_exam_weight', 'WTINTPRP': 'interview_weight', 'SDMVPSU': 'survey_psu', 'SDMVSTRA': 'survey_stratum'}

Neue Shape: (7437, 364)


,SEQN,age_years,income_poverty_ratio,mec_exam_weight,interview_weight,survey_psu,survey_stratum,gender,ethnicity,education,marital_status,pregnancy_status,country_of_birth,calories,protein,carbs,fat,iron,vitamin_b12,vitamin_d,folate,magnesium,zinc,height_cm,weight_kg,bmi,waist_cm,hip_cm,sbp_1,sbp_2,sbp_3,dbp_1,dbp_2,dbp_3,pulse_1,pulse_2,pulse_3,liver_cap_dbm,liver_stiffness_kpa,liver_exam_status,liver_valid_measures,liver_stiffness_iqr_ratio,oral_gum_problem_yesno,oral_decayed_teeth_yesno,oral_hygiene_issue_yesno,any_implant,any_root_caries,any_root_restoration_caries,fasting_hours_part,fasting_minutes_part,fasting_glucose_mg_dl,insulin_uU_ml,total_cholesterol_mg_dl,hdl_cholesterol_mg_dl,triglycerides_mg_dl,ferritin_ng_ml,serum_iron_ug_dl,tibc_ug_dl,transferrin_saturation_pct,transferrin_receptor_mg_l,uacr_mg_g,serum_creatinine_mg_dl,bun_mg_dl,alt_u_l,ast_u_l,ggt_u_l,alp_u_l,total_bilirubin_mg_dl,serum_albumin_g_dl,alq111___ever_had_a_drink_of_any_kind_of_alcohol,alq121___past_12_mo_how_often_drink_alcoholic_bev,alq130___avg_#_alcoholic_drinks/day___past_12_mos,alq142___#_days_have_4_or_5_drinks/past_12_mos,alq270___#_times_4_5_drinks_in_2hrs/past_12_mos,alq280___#_times_8+_drinks_in_1_day/past_12_mos,alq290___#_times_12+_drinks_in_1_day/past_12_mos,alq151___ever_have_4/5_or_more_drinks_every_day?,alq170___past_30_days_#_times_4_5_drinks_on_an_oc,bpq020___ever_told_you_had_high_blood_pressure,bpq030___told_had_high_blood_pressure___2+_times,bpd035___age_told_had_hypertension,bpq040a___taking_prescription_for_hypertension,bpq050a___now_taking_prescribed_medicine_for_hbp,bpq080___doctor_told_you___high_cholesterol_level,bpq060___ever_had_blood_cholesterol_checked,bpq070___when_blood_cholesterol_last_checked,bpq090d___told_to_take_prescriptn_for_cholesterol,bpq100d___now_taking_prescribed_medicine,cdq001___sp_ever_had_pain_or_discomfort_in_chest,cdq002___sp_get_it_walking_uphill_or_in_a_hurry,cdq003___during_an_ordinary_pace_on_level_ground,cdq004___if_so_does_sp_continue_or_slow_down,cdq005___does_standing_relieve_pain/discomfort,cdq006___how_soon_is_the_pain_relieved,cdq009a___pain_in_right_arm,cdq009b___pain_in_right_chest,cdq009c___pain_in_neck,cdq009d___pain_in_upper_sternum,cdq009e___pain_in_lower_sternum,cdq009f___pain_in_left_chest,...,rxqseen___medicine_container_seen_by_interviewer,rxddays___number_of_days_taken_medicine,rxdrsc1___icd_10_cm_code_1,rxdrsc2___icd_10_cm_code_2,rxdrsc3___icd_10_cm_code_3,rxdrsd1___icd_10_cm_code_1_description,rxdrsd2___icd_10_cm_code_2_description,rxdrsd3___icd_10_cm_code_3_description,rxdcount___number_of_prescription_medicines_taken,slq300___usual_sleep_time_on_weekdays_or_workdays,slq310___usual_wake_time_on_weekdays_or_workdays,sld012___sleep_hours___weekdays_or_workdays,slq320___usual_sleep_time_on_weekends,slq330___usual_wake_time_on_weekends,sld013___sleep_hours___weekends,slq030___how_often_do_you_snore?,slq040___how_often_do_you_snort_or_stop_breathing,slq050___ever_told_doctor_had_trouble_sleeping?,smq020___smoked_at_least_100_cigarettes_in_life,smd030___age_started_smoking_cigarettes_regularly,smq040___do_you_now_smoke_cigarettes?,smq050q___how_long_since_quit_smoking_cigarettes,smq050u___unit_of_measure_(day/week/month/year),smd057___#_cigarettes_smoked_per_day_when_quit,smq078___how_soon_after_waking_do_you_smoke,smd641___#_days_smoked_cigs_during_past_30_days,smd650___avg_#_cigarettes/day_during_past_30_days,smd100fl___cigarette_filter_type,smd100mn___cigarette_menthol_indicator,smq670___tried_to_quit_smoking,smq621___cigarettes_smoked_in_entire_life,smd630___age_first_smoked_whole_cigarette,smaquex2___questionnaire_mode_flag,whd010___current_self_reported_height_(inches),whd020___current_self_reported_weight_(pounds),whq030___how_do_you_consider_your_weight,"whq040___like_to_weigh_more,_less_or_same",whd050___self_reported_weight___1_yr_ago_(pounds),whq060___weight_change_intentional,whq070___tried_to_lose_weight_in_past_year,whd080a___ate_less_to_lose_weight,whd080b___switc

In [9]:
output_path = Path("../data/processed/nhanes_merged_adults_cleaned.csv")
df_clean.to_csv(output_path, index=False)
print(f"Gespeichert unter: {output_path}")

Gespeichert unter: ..\data\processed\nhanes_merged_adults_cleaned.csv


## Domain Mapping

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE_PATH = Path("../data/processed")

SOURCE_FILES = {
    "demographics": BASE_PATH / "demo_all_adults.csv",
    "dietary": BASE_PATH / "dietary_clean.csv",
    "examination": BASE_PATH / "examination_clean.csv",
    "laboratory": BASE_PATH / "laboratory_clean.csv",
    "questionnaire": BASE_PATH / "merged_questionnaire.csv",
}

In [11]:
def normalize_colname(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

In [12]:
catalog_rows = []

for domain, path in SOURCE_FILES.items():
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    
    for col in cols:
        catalog_rows.append({
            "source_domain": domain,
            "source_file": path.name,
            "source_column": col,
            "source_column_norm": normalize_colname(col)
        })

source_catalog = pd.DataFrame(catalog_rows)

display(source_catalog.head())
print("Anzahl Katalog-Zeilen:", len(source_catalog))

,source_domain,source_file,source_column,source_column_norm
0,demographics,demo_all_adults.csv,SEQN,seqn
1,demographics,demo_all_adults.csv,RIAGENDR,riagendr
2,demographics,demo_all_adults.csv,RIDAGEYR,ridageyr
3,demographics,demo_all_adults.csv,RIDRETH3,ridreth3
4,demographics,demo_all_adults.csv,RIDRETH1,ridreth1


Anzahl Katalog-Zeilen: 375


In [13]:
duplicate_source_cols = (
    source_catalog.groupby("source_column_norm")
    .agg(
        n_sources=("source_file", "nunique"),
        source_files=("source_file", lambda x: " | ".join(sorted(set(x)))),
        domains=("source_domain", lambda x: " | ".join(sorted(set(x))))
    )
    .reset_index()
)

duplicates_only = duplicate_source_cols[duplicate_source_cols["n_sources"] > 1]
display(duplicates_only.head(20))

,source_column_norm,n_sources,source_files,domains
294,seqn,5,demo_all_adults.csv | dietary_clean.csv | examination_clean.csv | laboratory_clean.csv | merged_questionnaire.csv,demographics | dietary | examination | laboratory | questionnaire


In [14]:
def dataset_overview(dataframe):
    overview = pd.DataFrame({
        "column": dataframe.columns,
        "dtype": dataframe.dtypes.astype(str).values,
        "missing_n": dataframe.isna().sum().values,
        "missing_pct": (dataframe.isna().mean().values * 100).round(2),
        "n_unique": dataframe.nunique(dropna=True).values
    })
    return overview.sort_values("missing_pct", ascending=False)

overview_clean = dataset_overview(df_clean)
display(overview_clean.head(30))

,column,dtype,missing_n,missing_pct,n_unique
167,mcq151___age_in_years_at_first_menstrual_period,float64,7437,100.00,0
295,smd630___age_first_smoked_whole_cigarette,float64,7437,100.00,0
294,smq621___cigarettes_smoked_in_entire_life,float64,7437,100.00,0
166,mcq149___menstrual_periods_started_yet?,float64,7437,100.00,0
358,medication_22,str,7435,99.97,2
357,medication_21,str,7435,99.97,2
356,medication_20,str,7434,99.96,3
355,medication_19,str,7432,99.93,5
203,mcq230c___what_kind_of_cancer_third_mention,float64,7432,99.93,4
189,mcq510b___liver_condition_non_alcoholic_fatty_liver,float64,7431,99.92,1


In [15]:
overview["column_norm"] = overview["column"].apply(normalize_colname)

source_lookup = (
    source_catalog.groupby("source_column_norm")
    .agg(
        source_domain=("source_domain", lambda x: " | ".join(sorted(set(x)))),
        source_file=("source_file", lambda x: " | ".join(sorted(set(x)))),
        source_column=("source_column", lambda x: " | ".join(sorted(set(x))))
    )
    .reset_index()
)

overview = overview.merge(
    source_lookup,
    left_on="column_norm",
    right_on="source_column_norm",
    how="left"
)

overview["domain"] = overview["source_domain"]
overview["mapping_method"] = np.where(
    overview["domain"].notna(),
    "exact_header_match",
    pd.NA
)

display(overview.head(20))

,column,dtype,missing_n,missing_pct,n_unique,column_norm,source_column_norm,source_domain,source_file,source_column,domain,mapping_method
0,smq621___cigarettes_smoked_in_entire_life,float64,7437,100.00,0,smq621_cigarettes_smoked_in_entire_life,smq621_cigarettes_smoked_in_entire_life,questionnaire,merged_questionnaire.csv,smq621___cigarettes_smoked_in_entire_life,questionnaire,exact_header_match
1,smd630___age_first_smoked_whole_cigarette,float64,7437,100.00,0,smd630_age_first_smoked_whole_cigarette,smd630_age_first_smoked_whole_cigarette,questionnaire,merged_questionnaire.csv,smd630___age_first_smoked_whole_cigarette,questionnaire,exact_header_match
2,mcq151___age_in_years_at_first_menstrual_period,float64,7437,100.00,0,mcq151_age_in_years_at_first_menstrual_period,mcq151_age_in_years_at_first_menstrual_period,questionnaire,merged_questionnaire.csv,mcq151___age_in_years_at_first_menstrual_period,questionnaire,exact_header_match
3,mcq149___menstrual_periods_started_yet?,float64,7437,100.00,0,mcq149_menstrual_periods_started_yet,mcq149_menstrual_periods_started_yet,questionnaire,merged_questionnaire.csv,mcq149___menstrual_periods_started_yet?,questionnaire,exact_header_match
4,medication_22,str,7435,99.97,2,medication_22,medication_22,questionnaire,merged_questionnaire.csv,medication_22,questionnaire,exact_header_match
5,medication_21,str,7435,99.97,2,medication_21,medication_21,questionnaire,merged_questionnaire.csv,medication_21,questionnaire,exact_header_match
6,medication_20,str,7434,99.96,3,medication_20,medication_20,questionnaire,merged_questionnaire.csv,medication_20,questionnaire,exact_header_match
7,mcq230c___what_kind_of_cancer_third_mention,float64,7432,99.93,4,mcq230c_what_kind_of_cancer_third_mention,mcq230c_what_kind_of_cancer_third_mention,questionnaire,merged_questionnaire.csv,mcq230c___what_kind_of_cancer_third_mention,questionnaire,exact_header_match
8,medication_19,str,7432,99.93,5,medication_19,medication_19,questionnaire,merged_questionnaire.csv,medication_19,questionnaire,exact_header_match
9,mcq510b___liver_condition_non_alcoholic_fatty_liver,float64,7431,99.92,1,mcq510b_liver_condition_non_alcoholic_fatty_liver,mcq510b_liver_condition_non_alcoholic_fatty_liver,questionnaire,merged_questionnaire.csv,mcq510b___liver_condition_non_alcoholic_fatty_liver,questionnaire,exact_header_match


In [16]:
overview.to_csv("03_overview_with_domain_mapping.csv", index=False)
print("Saved: 03_overview_with_domain_mapping.csv")

Saved: 03_overview_with_domain_mapping.csv


### doubles: 

In [17]:
demographic_cols = overview[
    overview["domain"].str.contains("demographics", na=False)
]

display(demographic_cols)

,column,dtype,missing_n,missing_pct,n_unique,column_norm,source_column_norm,source_domain,source_file,source_column,domain,mapping_method
146,RIDEXPRG,float64,5563,74.80,3,ridexprg,ridexprg,demographics,demo_all_adults.csv,RIDEXPRG,demographics,exact_header_match
147,pregnancy_status,str,5563,74.80,3,pregnancy_status,pregnancy_status,demographics,demo_all_adults.csv,pregnancy_status,demographics,exact_header_match
235,INDFMPIR,float64,1118,15.03,460,indfmpir,indfmpir,demographics,demo_all_adults.csv,INDFMPIR,demographics,exact_header_match
281,DMDEDUC2,float64,461,6.20,7,dmdeduc2,dmdeduc2,demographics,demo_all_adults.csv,DMDEDUC2,demographics,exact_header_match
282,DMDMARTZ,float64,461,6.20,4,dmdmartz,dmdmartz,demographics,demo_all_adults.csv,DMDMARTZ,demographics,exact_header_match
295,education,str,461,6.20,7,education,education,demographics,demo_all_adults.csv,education,demographics,exact_header_match
296,marital_status,str,461,6.20,4,marital_status,marital_status,demographics,demo_all_adults.csv,marital_status,demographics,exact_header_match
318,SDMVPSU,float64,0,0.00,3,sdmvpsu,sdmvpsu,demographics,demo_all_adults.csv,SDMVPSU,demographics,exact_header_match
319,SDMVSTRA,float64,0,0.00,24,sdmvstra,sdmvstra,demographics,demo_all_adults.csv,SDMVSTRA,demographics,exact_header_match
320,gender,str,0,0.00,2,gender,gender,demographics,demo_all_adults.csv,gender,demographics,exact_header_match


## Missingness Audit

In [18]:
high_missing = overview.sort_values("missing_pct", ascending=False)
display(high_missing.head(50))

,column,dtype,missing_n,missing_pct,n_unique,column_norm,source_column_norm,source_domain,source_file,source_column,domain,mapping_method
0,smq621___cigarettes_smoked_in_entire_life,float64,7437,100.00,0,smq621_cigarettes_smoked_in_entire_life,smq621_cigarettes_smoked_in_entire_life,questionnaire,merged_questionnaire.csv,smq621___cigarettes_smoked_in_entire_life,questionnaire,exact_header_match
1,smd630___age_first_smoked_whole_cigarette,float64,7437,100.00,0,smd630_age_first_smoked_whole_cigarette,smd630_age_first_smoked_whole_cigarette,questionnaire,merged_questionnaire.csv,smd630___age_first_smoked_whole_cigarette,questionnaire,exact_header_match
2,mcq151___age_in_years_at_first_menstrual_period,float64,7437,100.00,0,mcq151_age_in_years_at_first_menstrual_period,mcq151_age_in_years_at_first_menstrual_period,questionnaire,merged_questionnaire.csv,mcq151___age_in_years_at_first_menstrual_period,questionnaire,exact_header_match
3,mcq149___menstrual_periods_started_yet?,float64,7437,100.00,0,mcq149_menstrual_periods_started_yet,mcq149_menstrual_periods_started_yet,questionnaire,merged_questionnaire.csv,mcq149___menstrual_periods_started_yet?,questionnaire,exact_header_match
4,medication_22,str,7435,99.97,2,medication_22,medication_22,questionnaire,merged_questionnaire.csv,medication_22,questionnaire,exact_header_match
5,medication_21,str,7435,99.97,2,medication_21,medication_21,questionnaire,merged_questionnaire.csv,medication_21,questionnaire,exact_header_match
6,medication_20,str,7434,99.96,3,medication_20,medication_20,questionnaire,merged_questionnaire.csv,medication_20,questionnaire,exact_header_match
7,mcq230c___what_kind_of_cancer_third_mention,float64,7432,99.93,4,mcq230c_what_kind_of_cancer_third_mention,mcq230c_what_kind_of_cancer_third_mention,questionnaire,merged_questionnaire.csv,mcq230c___what_kind_of_cancer_third_mention,questionnaire,exact_header_match
8,medication_19,str,7432,99.93,5,medication_19,medication_19,questionnaire,merged_questionnaire.csv,medication_19,questionnaire,exact_header_match
9,mcq510b___liver_condition_non_alcoholic_fatty_liver,float64,7431,99.92,1,mcq510b_liver_condition_non_alcoholic_fatty_liver,mcq510b_liver_condition_non_alcoholic_fatty_liver,questionnaire,merged_questionnaire.csv,mcq510b___liver_condition_non_alcoholic_fatty_liver,questionnaire,exact_header_match


In [19]:
bins = [0, 5, 20, 50, 80, 95, 100]
labels = ["0-5%", "5-20%", "20-50%", "50-80%", "80-95%", "95-100%"]

overview["missing_bucket"] = pd.cut(
    overview["missing_pct"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

display(overview["missing_bucket"].value_counts().sort_index())

missing_bucket
0-5%       69
5-20%      79
20-50%     17
50-80%     70
80-95%     64
95-100%    72
Name: count, dtype: int64

## What is already done

- dataset overview
- missing-value overview
- readable renaming for key demographic columns
- source-domain mapping
- cleaned export to csv

In [27]:
(df_clean["SEQN"] == df_clean["SEQN.1"]).all()

np.True_

In [28]:
df_clean[["SEQN", "SEQN.1"]].head()

,SEQN,SEQN.1
0,109266,109266
1,109267,109267
2,109268,109268
3,109271,109271
4,109273,109273


### Cleaning Review: Work Completed So Far

At this stage, a substantial part of the initial data preparation has already been completed.

The existing workflow includes:
- a general dataset overview (`dtype`, missing values, number of unique values),
- renaming of selected demographic variables into more readable feature names,
- mapping each variable back to its original NHANES source domain,
- exporting a cleaned merged dataset as `df_clean`.

This means the next phase should focus less on rebuilding the preprocessing pipeline and more on validating the cleaned dataset and identifying remaining technical issues before continuing the EDA.

One issue identified during inspection is the presence of both `SEQN` and `SEQN.1`, which suggests a duplicated identifier column introduced during merging.  
The next step is to verify whether both columns contain the same participant ID values.


In [29]:
df_clean = df_clean.drop(columns=["SEQN.1"])

In [30]:
df_clean.shape

(7437, 363)

In [32]:
# how many columns per domain?
overview["domain"].value_counts()

domain
questionnaire                                                        295
examination                                                           25
laboratory                                                            21
demographics                                                          19
dietary                                                               10
demographics | dietary | examination | laboratory | questionnaire      1
Name: count, dtype: int64

In [33]:
overview.groupby("domain")["missing_pct"].mean().sort_values(ascending=False)

domain
questionnaire                                                        59.098305
laboratory                                                           22.161429
examination                                                          21.631200
dietary                                                              14.950000
demographics                                                          9.970000
demographics | dietary | examination | laboratory | questionnaire     0.000000
Name: missing_pct, dtype: float64

### Missingness by Data Domain

Since the merged dataset contains variables originating from several NHANES modules
(demographics, dietary, examination, laboratory, and questionnaire), it is useful to
analyze missing values at the domain level.

This helps determine which data sources are most complete and which contain large
amounts of structural missingness (for example due to questionnaire skip logic).

```python
overview.groupby("domain")["missing_pct"].mean().sort_values(ascending=False)

In [34]:
zero_var = df_clean.nunique(dropna=True) <= 1
zero_var_cols = df_clean.columns[zero_var]

len(zero_var_cols), list(zero_var_cols)[:10]

(41,
 ['cdq009a___pain_in_right_arm',
  'cdq009b___pain_in_right_chest',
  'cdq009c___pain_in_neck',
  'cdq009d___pain_in_upper_sternum',
  'cdq009e___pain_in_lower_sternum',
  'cdq009f___pain_in_left_chest',
  'cdq009g___pain_in_left_arm',
  'mcq149___menstrual_periods_started_yet?',
  'mcq151___age_in_years_at_first_menstrual_period',
  'mcq510b___liver_condition_non_alcoholic_fatty_liver'])

In [35]:
very_sparse = overview[overview["missing_pct"] > 95]
very_sparse[["column", "missing_pct", "domain"]].head(20)

,column,missing_pct,domain
0,smq621___cigarettes_smoked_in_entire_life,100.00,questionnaire
1,smd630___age_first_smoked_whole_cigarette,100.00,questionnaire
2,mcq151___age_in_years_at_first_menstrual_period,100.00,questionnaire
3,mcq149___menstrual_periods_started_yet?,100.00,questionnaire
4,medication_22,99.97,questionnaire
5,medication_21,99.97,questionnaire
6,medication_20,99.96,questionnaire
7,mcq230c___what_kind_of_cancer_third_mention,99.93,questionnaire
8,medication_19,99.93,questionnaire
9,mcq510b___liver_condition_non_alcoholic_fatty_liver,99.92,questionnaire


In [36]:
corr = df_clean.corr(numeric_only=True)

corr["fatigue_binary"].sort_values(ascending=False).head(15)

fatigue_binary                                       1.000000
dpq040___feeling_tired_or_having_little_energy       0.833404
rhq020___age_range_at_first_menstrual_period         0.481932
cluster                                              0.225876
huq010___general_health_condition                    0.206760
kiq430___how_frequently_does_this_occur?             0.187132
kiq450___how_frequently_does_this_occur?             0.184430
kiq050___how_much_did_urine_leakage_bother_you?      0.181710
kiq005___how_often_have_urinary_leakage?             0.180843
mcd180f___age_when_told_you_had_a_stroke             0.171939
med_count                                            0.171644
rxdcount___number_of_prescription_medicines_taken    0.171306
heq040___ever_prescribed_meds_treat_hepatitis_c?     0.165780
kiq470___how_frequently_does_this_occur?             0.155550
kiq052___how_much_were_daily_activities_affected?    0.151362
Name: fatigue_binary, dtype: float64

In [37]:
corr["fatigue_binary"].sort_values().head(15)

nan_count                                          -0.213764
slq050___ever_told_doctor_had_trouble_sleeping?    -0.205516
cdq001___sp_ever_had_pain_or_discomfort_in_chest   -0.194195
mcq520___abdominal_pain_during_past_12_months?     -0.164046
cdq010___shortness_of_breath_on_stairs/inclines    -0.149784
huq090___seen_mental_health_professional/past_yr   -0.136151
heq020___ever_prescribed_meds_treat_hepatitis_b?   -0.124106
mcq230b___what_kind_of_cancer_second_mention       -0.114307
kiq042___leak_urine_during_physical_activities?    -0.109179
rxduse___taken_prescription_medicine,_past_month   -0.106831
whq030___how_do_you_consider_your_weight           -0.103236
kiq044___urinated_before_reaching_the_toilet?      -0.098168
mcq080___doctor_ever_said_you_were_overweight      -0.097589
income_poverty_ratio                               -0.097143
mcq170l___still_have_liver_condition               -0.095632
Name: fatigue_binary, dtype: float64

### Initial Feature Screening

To guide the EDA and prepare for later modeling, we performed three initial screening steps:

1. identify columns with no variation,
2. inspect highly sparse columns,
3. check early associations with the target variable `fatigue_binary`.

#### 1. Zero-variance features

Columns with only one unique value do not contain predictive information and are strong candidates for removal.

In our dataset, this screening identified **41 zero-variance columns**.  
Examples include several pain-location variables and some highly sparse questionnaire fields.

#### 2. Highly sparse features

We also reviewed columns with more than 95% missing values.  
Most of these belong to the **questionnaire** domain and likely reflect **survey skip logic** or rare conditions rather than simple data quality problems.

This means high missingness alone is **not sufficient reason to drop a feature**.  
Some sparse variables may still be medically meaningful and should be reviewed in context.

#### 3. Early target associations

A first correlation scan with `fatigue_binary` revealed several highly associated variables.  
However, some of these appear to be problematic:

- `dpq040___feeling_tired_or_having_little_energy` is likely too close to the target itself and may introduce **target leakage**
- `cluster` and `nan_count` appear to be **technical or engineered columns**, not real health predictors

#### Conclusion

Before continuing with broader EDA, the next step is to identify and exclude
**leakage features** and **technical artifact columns**, so that later analysis focuses only on meaningful health-related predictors.

In [38]:
pd.crosstab(df_clean["dpq040___feeling_tired_or_having_little_energy"], df_clean["fatigue_binary"], dropna=False)

fatigue_binary,0,1
dpq040___feeling_tired_or_having_little_energy,,
0.0,3171,0
1.0,2180,0
2.0,0,613
3.0,0,499
7.0,0,2
9.0,0,3
NaN,969,0


In [39]:
df_clean = df_clean.drop(columns=["dpq040___feeling_tired_or_having_little_energy"])

### Target Leakage Check

During the correlation analysis, the variable  
`dpq040___feeling_tired_or_having_little_energy` showed an extremely strong relationship with the target variable `fatigue_binary`.

To investigate this, we examined the cross-tabulation between the two variables.

```python
pd.crosstab(df_clean["dpq040___feeling_tired_or_having_little_energy"], df_clean["fatigue_binary"], dropna=False)

In [40]:
df_clean[["cluster", "nan_count", "nan_group"]].head()

,cluster,nan_count,nan_group
0,1,172,151-200
1,1,201,201-250
2,1,216,201-250
3,3,166,151-200
4,3,172,151-200


In [41]:
df_clean = df_clean.drop(columns=["cluster", "nan_count", "nan_group"])

### Removal of Technical Artifact Features

During inspection of the dataset, three variables were identified that do not represent participant health information:

- `cluster`
- `nan_count`
- `nan_group`

These variables were created during preprocessing to track missing values and group observations based on the number of missing features.

Example:

| cluster | nan_count | nan_group |
|---|---|---|
| 1 | 172 | 151–200 |
| 3 | 166 | 151–200 |

#### Interpretation

These features describe **data completeness**, not medical characteristics of participants.

Including them in modeling could lead to **artifact-based predictions**, where the model learns patterns in missingness rather than real health signals.

#### Action

To avoid dataset artifact leakage, these preprocessing artifact variables were removed from the dataset.

In [42]:
df_clean.shape

(7437, 359)

### Exploring Physiological Drivers of Fatigue

After removing leakage and artifact variables, we focus on examining
relationships between fatigue and medically meaningful physiological variables.

These include:

- demographic variables (e.g., age)
- body composition (BMI)
- dietary intake (macronutrients and micronutrients)
- laboratory markers related to anemia and metabolic health

Examples include:

- iron, ferritin, and transferrin saturation (anemia indicators)
- vitamin B12 and folate
- glucose and insulin (metabolic markers)
- cholesterol and triglycerides
- macronutrient intake (calories, protein, fat, carbohydrates)

To explore potential relationships with fatigue, we compute correlations
between these variables and the target variable `fatigue_binary`.

This step helps identify early signals of physiological factors
that may contribute to fatigue and guides deeper analysis in later stages.

In [43]:
core_cols = [
    "age_years",
    "bmi",
    "calories",
    "protein",
    "carbs",
    "fat",
    "iron",
    "vitamin_b12",
    "vitamin_d",
    "folate",
    "magnesium",
    "zinc",
    "ferritin_ng_ml",
    "serum_iron_ug_dl",
    "transferrin_saturation_pct",
    "fasting_glucose_mg_dl",
    "insulin_uU_ml",
    "total_cholesterol_mg_dl",
    "hdl_cholesterol_mg_dl",
    "triglycerides_mg_dl",
]

df_clean[core_cols + ["fatigue_binary"]].corr()["fatigue_binary"].sort_values(ascending=False)

fatigue_binary                1.000000
bmi                           0.099088
insulin_uU_ml                 0.054041
age_years                     0.025452
fasting_glucose_mg_dl         0.012439
total_cholesterol_mg_dl       0.005072
carbs                        -0.006211
triglycerides_mg_dl          -0.006558
vitamin_b12                  -0.008276
hdl_cholesterol_mg_dl        -0.008660
fat                          -0.011761
calories                     -0.015360
zinc                         -0.023773
vitamin_d                    -0.032374
ferritin_ng_ml               -0.037549
protein                      -0.038097
iron                         -0.043741
serum_iron_ug_dl             -0.046377
transferrin_saturation_pct   -0.057985
folate                       -0.059586
magnesium                    -0.065831
Name: fatigue_binary, dtype: float64

### Physiological Variables and Fatigue

To explore potential biological drivers of fatigue, we examined correlations between the target variable `fatigue_binary` and a set of core physiological variables including:

- body composition (BMI)
- nutritional intake
- micronutrient levels
- metabolic markers

```python
df_clean[core_cols + ["fatigue_binary"]].corr()["fatigue_binary"].sort_values(ascending=False)

### Data Type Inspection

Before performing additional feature engineering, we inspected the data types of all variables.

Large merged health datasets often contain inconsistencies such as:

- categorical variables stored as numeric codes
- numeric variables stored as strings
- mixed encodings due to merging multiple datasets

To identify potential issues, we examined:

1. columns stored as object (string) types
2. numeric columns with very small numbers of unique values, which may represent categorical variables

```python
df_clean.select_dtypes(include="object").columns

In [44]:
df_clean.select_dtypes(include="object").columns

C:\Users\Inbal\AppData\Local\Temp\ipykernel_19184\432708615.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_clean.select_dtypes(include="object").columns


Index(['gender', 'ethnicity', 'education', 'marital_status',
       'pregnancy_status', 'country_of_birth', 'rxddrug___generic_drug_name',
       'rxddrgid___generic_drug_code', 'rxdrsc1___icd_10_cm_code_1',
       'rxdrsc2___icd_10_cm_code_2', 'rxdrsc3___icd_10_cm_code_3',
       'rxdrsd1___icd_10_cm_code_1_description',
       'rxdrsd2___icd_10_cm_code_2_description',
       'rxdrsd3___icd_10_cm_code_3_description',
       'slq300___usual_sleep_time_on_weekdays_or_workdays',
       'slq310___usual_wake_time_on_weekdays_or_workdays',
       'slq320___usual_sleep_time_on_weekends',
       'slq330___usual_wake_time_on_weekends', 'medication_1', 'medication_2',
       'medication_3', 'medication_4', 'medication_5', 'medication_6',
       'medication_7', 'medication_8', 'medication_9', 'medication_10',
       'medication_11', 'medication_12', 'medication_13', 'medication_14',
       'medication_15', 'medication_16', 'medication_17', 'medication_18',
       'medication_19', 'medication_20'

In [45]:
small_unique = df_clean.select_dtypes(include=["float64","int64"]).nunique()
small_unique[small_unique <= 10].sort_values()

mcq151___age_in_years_at_first_menstrual_period          0
mcq149___menstrual_periods_started_yet?                  0
smq621___cigarettes_smoked_in_entire_life                0
smd630___age_first_smoked_whole_cigarette                0
cdq009g___pain_in_left_arm                               1
cdq009e___pain_in_lower_sternum                          1
cdq009f___pain_in_left_chest                             1
cdq009a___pain_in_right_arm                              1
mcq510f___liver_condition_other                          1
mcq510c___liver_condition_alcoholic_liver_disease        1
mcq510e___liver_condition_autoimmune                     1
mcq510d___liver_condition_hepatitis                      1
cdq009d___pain_in_upper_sternum                          1
mcq510b___liver_condition_non_alcoholic_fatty_liver      1
cdq009c___pain_in_neck                                   1
cdq009b___pain_in_right_chest                            1
whd080k___took_laxatives_or_vomited                     

### Removal of Zero-Variance Features

To identify variables that cannot contribute meaningful information to the analysis,
we examined the number of unique values in each column.

```python
zero_var_cols = df_clean.columns[df_clean.nunique(dropna=True) <= 1]
len(zero_var_cols)

In [46]:
zero_var_cols = df_clean.columns[df_clean.nunique(dropna=True) <= 1]
len(zero_var_cols)

41

In [47]:
zero_var_cols[:20]

Index(['cdq009a___pain_in_right_arm', 'cdq009b___pain_in_right_chest',
       'cdq009c___pain_in_neck', 'cdq009d___pain_in_upper_sternum',
       'cdq009e___pain_in_lower_sternum', 'cdq009f___pain_in_left_chest',
       'cdq009g___pain_in_left_arm', 'mcq149___menstrual_periods_started_yet?',
       'mcq151___age_in_years_at_first_menstrual_period',
       'mcq510b___liver_condition_non_alcoholic_fatty_liver',
       'mcq510c___liver_condition_alcoholic_liver_disease',
       'mcq510d___liver_condition_hepatitis',
       'mcq510e___liver_condition_autoimmune',
       'mcq510f___liver_condition_other', 'rhq542a___hormone_pills_used',
       'rhq542b___hormone_patches_used',
       'rhq542c___hormone_cream/suppository/injection_used',
       'smq621___cigarettes_smoked_in_entire_life',
       'smd630___age_first_smoked_whole_cigarette',
       'smaquex2___questionnaire_mode_flag'],
      dtype='str')

In [48]:
df_clean = df_clean.drop(columns=zero_var_cols)

In [49]:
df_clean.shape

(7437, 318)

### Missing Values per Participant

Because the dataset combines multiple NHANES modules (dietary, laboratory,
examination, questionnaire), not every participant has measurements in all
domains.

To understand how sparse the dataset is at the participant level, we calculated
the number of missing values per row.

```python
df_clean.isna().sum(axis=1).describe()

In [50]:
df_clean.isna().sum(axis=1).describe()

count    7437.000000
mean      149.681727
std        28.526824
min        62.000000
25%       132.000000
50%       146.000000
75%       162.000000
max       251.000000
dtype: float64

### Highly Sparse Variables

We examined the columns with the highest proportion of missing values:

```python
df_clean.isna().mean().sort_values(ascending=False).head(20)

In [51]:
df_clean.isna().mean().sort_values(ascending=False).head(20)

medication_22                                       0.999731
medication_21                                       0.999731
medication_20                                       0.999597
medication_19                                       0.999328
mcq230c___what_kind_of_cancer_third_mention         0.999328
medication_18                                       0.999059
medication_17                                       0.998386
medication_16                                       0.997580
rhq020___age_range_at_first_menstrual_period        0.996773
medication_15                                       0.996638
rhq070___age_range_at_last_menstrual_period         0.996101
mcq230b___what_kind_of_cancer_second_mention        0.996101
medication_14                                       0.995294
medication_13                                       0.994218
rxdrsc3___icd_10_cm_code_3                          0.991529
rxdrsd3___icd_10_cm_code_3_description              0.991529
heq020___ever_prescribed

#### Interpretation

Many of the sparsest variables belong to the **questionnaire module** and represent
either conditional questions (asked only if a previous answer triggered them)
or rare medical conditions.

For example:
- medication slots (`medication_11`–`medication_22`) are empty for most participants
  because individuals typically take only a few medications.
- variables such as cancer mentions or disease diagnosis ages apply only to small
  subsets of participants.

Therefore, the high proportion of missing values is primarily caused by the
**survey’s conditional structure**, not by poor data quality.

At this stage these variables are **retained**, as some may still provide useful
signals for specific health conditions related to fatigue.

### Low-Cardinality Variables

We inspected variables with very small numbers of unique values.

```python
df_clean.nunique().sort_values().head(20)

In [52]:
df_clean.nunique().sort_values().head(20)

fatigue_binary                                      2
medication_22                                       2
medication_21                                       2
smq670___tried_to_quit_smoking                      2
whq060___weight_change_intentional                  2
whq070___tried_to_lose_weight_in_past_year          2
rhq200___now_breastfeeding_a_child?                 2
paq665___moderate_recreational_activities           2
paq650___vigorous_recreational_activities           2
mcq500___ever_told_you_had_any_liver_condition      2
mcq510a___liver_condition_fatty_liver               2
mcq050___emergency_care_visit_for_asthma/past_yr    2
kiq025___received_dialysis_in_past_12_months?       2
diq060u___unit_of_measure_(month/year)              2
cdq003___during_an_ordinary_pace_on_level_ground    2
cdq004___if_so_does_sp_continue_or_slow_down        2
alq111___ever_had_a_drink_of_any_kind_of_alcohol    2
bpq050a___now_taking_prescribed_medicine_for_hbp    2
oral_gum_problem_yesno      

### most important safe cleanup is now done:

✔ removed duplicate ID  
✔ removed target leakage  
✔ removed preprocessing artifacts  
✔ removed zero-variance features  
✔ inspected missingness  
✔ checked feature types  
✔ checked correlations  